In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics,ensemble
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import warnings
warnings.filterwarnings('ignore')

<div class="jumbotron">
    <h1 class="display-1">Classification</h1>
    <hr class="my-4">
    <p>Instructor: Dr. Yan Li</p>
</div>

> How can we tell whether a hotel reservation will be cancelled?

In [2]:
hotelDict = {'ID': range(10),
            'Repeated': ['Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes','No', 'No', 'No'],
            'Segment': ['Leisure', 'Business', 'Leisure', 'Business', 'Group', 'Business', 'Group', 'Leisure', 'Business', 'Leisure'],
            'LeadTime': [125, 100, 70, 120, 95, 60, 220, 85, 75, 90],
            'Cancelled': ['No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes']}

pd.DataFrame(hotelDict)

,ID,Repeated,Segment,LeadTime,Cancelled
0,0,Yes,Leisure,125,No
1,1,No,Business,100,No
2,2,No,Leisure,70,No
3,3,Yes,Business,120,No
4,4,No,Group,95,Yes
5,5,No,Business,60,No
6,6,Yes,Group,220,No
7,7,No,Leisure,85,Yes
8,8,No,Business,75,No
9,9,No,Leisure,90,Yes


## Basic Concepts

### Classification

- Given a record $(\boldsymbol{x}, y)$, where $y$ is the class attribute (or target attribute, the label) and $\boldsymbol{x}$ is the set of predictive attributes (features) of the record

- By learning a **target function** $f$, map each attribute set $\boldsymbol{x}$ to a pre-defined **class label** $y$

$$
y = f(\boldsymbol{x})
$$

- The target function is called the ***classification model***

- A model is good if it makes few mistakes. The simplest measure of a mistake is the **0-1 loss**:

$$ L(y,\hat{y}) = \begin{cases} 0, & \hat{y}=y\\ 1, & \hat{y}\ne y \end{cases} $$

- Building a classifier is therefore equivalent to **choosing the target function $f$ that keeps the total 0-1 loss small**.

#### Areas Where Classification Models Apply

- Well suited to datasets whose class attribute is **binary** or **nominal**

- Not suited to datasets whose class attribute is **ordinal** or **continuous**
    + because the order and magnitude of the labels is not taken into account

Task|Attribute set $\boldsymbol{x}$|Class attribute $y$
---|---|---
Classify e-mail|Features extracted from e-mail headers and content|Spam `or` Non-spam
Diagnose cancer cells|Features extracted from MRI scans|Malignant `or` Benign
Classify galaxies|Features from astronomical telescope images|Elliptical, spiral, `or` irregular galaxies

### General Approach to Building a Classification Model

#### Can we use the whole dataset strategy?

<center><img src="./img/classification/classificationWholeDataset.png" width=80%></center>

#### The Correct Approach to Classification

<center><img src="./img/classification/classificationProcess.png" width=80%></center>

#### Training Set and Test Set

- Training set: records with known class labels, used to build the classification model

- Test set: the set of records used to validate the classification rules

```python
from sklearn.model_selection import train_test_split
train_test_split(*arrays, test_size=0.25, random_state=None, shuffle=True)
```

- `*arrays`: the data sequences to split; can be `lists`, `numpy.arrays`, or `pandas.DataFrame` objects

- `test_size`: size of the test set
    + if `float`, a fraction in `[0,1]` of the original dataset
    + if `int`, the absolute number of test records
    + defaults to 0.25

- `random_state`: random seed
    - `int` in $[0,2^{32}-1]$
    - controls the random ordering before the split

- `shuffle`: whether to shuffle the data randomly before the split
    + defaults to `True`
    + **Why shuffle?**
        - The training set and the test set must be representative, independent random samples of the same underlying distribution.
        - If the records are ordered (e.g. sorted by class label, by time, or clustered in groups), the consecutive split would put one class (or one period) entirely in the training set and the other entirely in the test set.
        - Shuffling breaks this ordering so that each set has a similar mix of classes; otherwise the model would be evaluated on test records that do not resemble the training records, biasing the claimed accuracy.

- Return: the split training and test sets, with the same types and ordering as the input `*array`

#### Typical Classification Algorithms

+ k-nearest neighbor (instance-based / lazy learning)
+ **Decision tree** (ID3, C4.5, CART)
+ **Random forest** (ensemble of decision trees)
+ Naive Bayes (probabilistic)
+ Logistic regression (linear model)
+ Linear discriminant analysis / Quadratic discriminant analysis (LDA / QDA)
+ Neural network (MLP, CNN, RNN)
+ Support vector machine (kernel-based)
+ **Boosting: AdaBoost, Gradient Boosting, XGBoost**

## Decision Trees

### Definition

A **decision tree** is a classification model that looks like a flowchart.

- **Nodes** ask a question about a record, e.g. "Is the lead time longer than 77 days?"
- **Branches** are the possible answers to that question (Yes / No).
- **Leaves** give the final decision, e.g. "the reservation will be cancelled".

> To classify a new record, start at the top (root) node, answer the question, follow the matching branch, and keep going until you reach a leaf that tells you the class.

<center><img src="./img/classification/hotel_decision_tree_1.png" with=50%></center>

<center><img src="./img/classification/hotel_decision_tree_2.png"></center>

> For the same problem, the decision tree is not unique

### Terminolgies

<center><img src="./img/classification/hotel_decision_tree_concepts.png" with=60%></center>

- **Root node**
    + the topmost node; represents the whole training set and contains the first test condition

- **Internal node**
    + a test node that is neither the root nor a leaf; contains a test condition and passes records down a child branch

- **Leaf node**
    + also called a terminal node
    + holds a class label; records reaching it are assigned that class, so it performs no further test

### Constructing a Decision Tree

- **Hunt** algorithm
- CART
- ID3, C4.5
- SLIQ, SPRINT

#### Hunt's Algorithm

<center><img src="./img/classification/hunt_algorithm_flowchart.png" width=70%></center>

#### CART Algorithm


While Hunt's algorithm is the foundation, `scikit-learn` uses an optimized version of **CART** (Classification and Regression Trees):

- **Binary Splitting:** Every node splits into exactly two children.
- **Exhaustive Search:** Evaluates all features and all possible split points.
- **Impurity Measures:** Gini or Entropy for classification; MSE for regression.


**The CART Splitting Logic:**

<center><img src="./img/classification/cart_algorithm_flowchart.png" width=75%></center>

Issues to Consider in Building a Decision Tree

- How to choose a test condition?
    + Which attribute to use as the splitting condition?
    + How to choose the split point for each condition? i.e., how to evaluate the quality of a split

- How to stop tree growth?
    + until all records belong to the **same** class, or have **identical** attribute values
    + other methods

Choosing the test condition

#### The Core Idea: Impurity Reduction

<center><img src="./img/classification/impurity_reduction_concept.png" width=80%></center>

- **Before Split:** Parent node has high impurity (a mix of classes).
- **After Split:** Children nodes are purer (mostly one class).
- **Decision Rule:** The difference between the parent's impurity and the weighted average of the children's impurity is the **Gain**. The larger this gain, the better the test condition.

In [6]:
carDict = [
    range(1,21),
    ['Male']*6+['Female']*4+['Male']*4+['Female']*6,
    ['Family']+['Sports']*8+['Luxury']
    +['Family']*3+['Luxury']*7,
    ['Small']+['Medium']*2+['Large']+['ExtraLarge']*2+['Small']*2+['Medium']+['Large']*2+['ExtraLarge']+['Medium']+['ExtraLarge']+['Small']*2
    +['Medium']*3+['Large'],
    ['C0']*10+['C1']*10
]
cardf = pd.DataFrame(carDict,index=['ID', 'Gender','CarModel', 'CarSize', 'Category']).T

In [5]:
cardf

,ID,Gender,CarModel,CarSize,Category
0,1,Male,Family,Small,C0
1,2,Male,Sports,Medium,C0
2,3,Male,Sports,Medium,C0
3,4,Male,Sports,Large,C0
4,5,Male,Sports,ExtraLarge,C0
5,6,Male,Sports,ExtraLarge,C0
6,7,Female,Sports,Small,C0
7,8,Female,Sports,Small,C0
8,9,Female,Sports,Medium,C0
9,10,Female,Luxury,Large,C0


Which is the the best split for the above dataset?

<center><img src="./img/classification/test_conditions.png" width=100%></center>

- The best split is usually chosen according to the node ***degree of impurity***

- The lower the impurity, the more skewed the class distribution

### Impurity Measures

Let $p(i|t)$ be the proportion of records in node $t$ belonging to class $i$, with $c$ classes

- **Entropy**
Let $p(i|t)$ be the proportion of records in node $t$ belonging to class $i$, with $c$ classes

$$ \text{Entropy}(t)=-\sum_{i=0}^{c-1}p(i|t)\log_2p(i|t) $$

> ID3 and C4.5 use entropy to select the best split

- **Gini index**

$$ \text{Gini}(t)=1-\sum_{i=1}^{c-1}\left[p(i|t)\right]^2 $$

> CART uses the Gini index to select the best split
- each non-leaf node has two branches, forming a binary tree

Suppose a node contains only two classes (class 1 and class 2), with proportions $p_1$ and $p_2$, and $p_1+p_2=1$

In [ ]:
x = np.linspace(0.01,0.99)
x1 = np.ones(x.shape) - x
xE = -x*np.log2(x)-x1*np.log2(x1)
xG = 1-(np.power(x,2)+np.power(x1,2))
df = pd.DataFrame({'x':x,'Entropy':xE,'Gini':xG})
ax = df.plot(x='x',y='Entropy',kind='line',figsize=(12,6))
df.plot(x='x',y='Gini',kind='line',ax=ax)
ax.set(xlabel='$p_1$')

> Entropy and Gini are not the 0-1 loss itself, but **surrogate objectives**: the tree picks the split that most reduces impurity, hoping this also reduces the 0-1 loss. The same pattern returns in boosting, where we minimize a **chosen loss** directly.

Determining a Test Condition

- For a chosen test condition, compute the difference between the parent-node impurity (before the split) and the child-node impurity (after the split); the larger the difference, the better the condition

Implementing with `sklearn`

Building the Model

```python
from sklearn import tree
tree.DecisionTreeClassifier(criterion='gini')
```
- `criterion`: `str`, the impurity measure; either `gini` or `entropy`, defaults to `gini`

- Attributes of the fitted tree
    + `classes_`: array of class labels
    + `n_classes_`: `int`, number of classes
    + `tree_`: the fitted decision tree

- `feature_importances_`: the importance of each attribute, i.e. the normalized reduction in the `Gini` coefficient caused by each attribute

In [ ]:
from sklearn import tree
dtModel = tree.DecisionTreeClassifier()
dtModel

Training the Model

```python
dt.fit(X, y)
```
- `X`: input feature matrix, shape `[n_samples, n_features]`
- `y`: array of class labels, shape `[n_samples]`

In [ ]:
hotel_data=[['Yes','No','No','Yes','No','No','Yes','No','No','No'],
      ['Leisure','Business','Leisure','Business','Group','Business','Group','Leisure','Business','Leisure'],
      [125,100,70,120,95,60,220,85,75,90],
      ['No','No','No','No','Yes','No','No','Yes','No','Yes']]
trainData = pd.DataFrame(hotel_data, index=['Repeated','Segment','LeadTime','Cancelled']).T
trainData

In [ ]:
dtModel.fit(trainData.iloc[:,:-1], trainData.iloc[:,-1])

Transforming the Predictive Attributes

> `DecisionTreeClassifier` only supports numeric predictive attributes
- but places no requirement on the class labels

##### **One-Hot Encoding**: convert a nominal attribute into binary attributes

```python
pandas.get_dummies(data, columns=None)
```
- `data`: a `Series` or `DataFrame`
- `columns`: a `list` of column names to transform; by default all columns
- Return: a `DataFrame` of binarized attributes

In [ ]:
trainDataOH = pd.get_dummies(trainData, columns=['Repeated','Segment'])
trainData
trainDataOH
trainDataY = trainDataOH.pop('Cancelled')
trainDataOH

In [ ]:
dtModel.fit(trainDataOH, trainDataY)

Visualizing a Decision Tree

Outputting Decision Rules as Text

```python
tree.export_text(decision_tree, feature_names=None)
```
- `decision_tree`: the fitted decision tree
- `feature_names`: list of predictor attribute names

In [ ]:
print(tree.export_text(dtModel,feature_names=list(trainDataOH.columns)))

Rendering Decision Rules Graphically

```python
tree.plot_tree(decision_tree,max_depth=None,feature_names=None,class_names=None,filled=False,ax=None)
```
- `max_depth`: `int`, maximum depth to display
- `class_names`: `list` of class names, in ascending order of the numeric value of each class
- `filled`: fill the nodes with color
- `ax`: the `axis` of `matplotlib`; draws the tree on this axis

In [ ]:
_ = tree.plot_tree(dtModel,filled=True,feature_names=list(trainDataOH.columns),class_names=dtModel.classes_)

- To enlarge the tree, explicitly set the size of the axes

In [ ]:
figdt,axdt = plt.subplots(figsize=(25,15))
_ = tree.plot_tree(dtModel,filled=True,feature_names=list(trainDataOH.columns),class_names=dtModel.classes_,ax=axdt)

Predicting with a Decision Tree

```python
dt.predict(X)
```
- `X`: input feature matrix, shape `[n_samples, n_features]`
- Return: predicted classes, an array of shape `[n_samples]`

```python
dt.predict_proba(X)
```
- Return: predicted probability of belonging to each class, a matrix of shape `[n_samples, n_classes]`; the class order follows `dt.classes_`

In [ ]:
hotel_test_data = [['Yes','No','No','Yes','No'],
         ['Leisure','Business','Leisure','Business','Group'],
         [200,80,70,100,65]]
testData = pd.DataFrame(hotel_test_data,index=['Repeated','Segment','LeadTime']).T
testDataOH = pd.get_dummies(testData, columns=['Repeated','Segment'])
testDataOH

In [ ]:
dtModel.predict(testDataOH)
dtModel.predict_proba(testDataOH)
dtModel.classes_

## Hotel Booking Cancellation Classification

#### Reading the Data

In [ ]:
titRawDf = pd.read_csv('data/analysis/hotel_bookings.csv')
titRawDf.head()

In [ ]:
titDf = titRawDf.loc[:,[titRawDf.index.name,'is_repeated_guest','adults','lead_time','stays_in_weekend_nights','stays_in_week_nights','adr','previous_cancellations','is_canceled']]
titDf.set_index(titRawDf.index.name,inplace=True)
titDf.head()

#### Data Preprocessing

##### Drop rows containing missing values

In [ ]:
titDf.dropna(axis=0,how='any',inplace=True)

##### One-Hot Encoding

In [ ]:
titDf.dtypes

In [ ]:
titX = titDf.iloc[:,:-1]
titY = titDf['is_canceled']
titX.head()

In [ ]:
titXOH = pd.get_dummies(titX,columns=['adults','previous_cancellations'])
titXOH.head()

#### Splitting the Data into Training and Test Sets

```python
from sklearn.model_selection import train_test_split
train_test_split(*arrays, test_size=0.25, random_state=None)
```

- `*arrays`: the data sequences to split; can be `lists`, `numpy.arrays`, or `pandas.DataFrame` objects

- `test_size`: size of the test set
    + if `float`, a fraction in `[0,1]` of the original dataset
    + if `int`, the absolute number of test records
    + defaults to 0.25

- `random_state`: random seed
    - `int` in $[0,2^{32}-1]$
    - controls the random ordering before the split

- Return: the split training and test sets, with the same types and ordering as the input `*array`

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
titTrainX,titTestX,titTrainY,titTestY = train_test_split(titXOH,titY,random_state=100)
titTrainX
titTestX
titTrainY.value_counts()
titTestY.value_counts()

#### Building the Decision Tree Model

In [ ]:
titDt = tree.DecisionTreeClassifier(random_state=10)

#### Training the Model on the Training Set

In [ ]:
titDt.fit(titTrainX,titTrainY)

#### Visualizing the Decision Tree

In [ ]:
print(tree.export_text(titDt,feature_names=list(titTrainX.columns)))

In [ ]:
figtit,axtit = plt.subplots(figsize=(20,15))
_ = tree.plot_tree(titDt,feature_names=list(titTrainX.columns),class_names=['Not canceled','Canceled'],filled=True,ax=axtit,max_depth=3)

In [ ]:
titDot = tree.export_graphviz(titDt,feature_names=list(titTrainX.columns),class_names=['Not canceled','Canceled'],filled=True)
titDtGraph = graphviz.Source(titDot)
titDtGraph

Saving the decision tree image

```python
graph.render(filename=None, directory=None, cleanup=False, format=None)
```
- `filename`: the name of the file to be saved
- `directory`: the path of the directory in which the file is saved
- `cleanup`: delete the intermediate files
- `format`: the image format to save, e.g. `png`, `pdf`

In [ ]:
titDtGraph.render(filename='titDt',directory='./img/classification/',cleanup=True,format='pdf')

## Measuring Classification Performance

### Confusion Matrix
- A matrix of the correct and incorrect classifications made by the classification model

<center><img src="./img/classification/PosNeg.svg" width=60%></center>

- **Accuracy score**

$$
\text{accuracy}=\frac{\mathrm{Number\ of\ correct\ predictions}}{\mathrm{Total\ predictions}}=\frac{TP+TN}{TP+FP+FN+TN}
$$

- **Precision score**
$$precision = \begin{cases}
\frac{TP}{TP+FP},\quad \text{positive class}\\
\frac{TN}{TN+FN},\quad \text{negative class}
\end{cases}$$

- The ability to make no incorrect classification for the samples of a given class

- **Recall score**
$$recall = \begin{cases}
\frac{TP}{TP+FN},\quad \text{positive class}\\
\frac{TN}{TN+FP},\quad \text{negative class}
\end{cases}$$

- The ability to identify all of the samples that truly belong to a given class

- `estimator`: the trained classifier
- `X`: the predictive attributes
- `value_formats`: the display format of the numbers

In [ ]:
from sklearn import metrics

In [ ]:
titTrainYPre = titDt.predict(titTrainX)

In [ ]:
metrics.confusion_matrix(titTrainY, titTrainYPre)
metrics.plot_confusion_matrix(titDt,titTrainX,titTrainY,values_format='.0f')

### Accuracy Score

```python
from sklearn import metrics
metrics.accuracy_score(y_true, y_pred)
```
- `y_true`: the array of true class labels
- `y_pred`: the array of class labels predicted by the classification model

> Bridges the metric back to loss: **accuracy = $1 -$ (mean 0-1 loss)** — the fraction of predictions the model gets right.

In [ ]:
metrics.accuracy_score(titTrainY,titTrainYPre)

### Recall Score

```python
metrics.recall_score(y_true, y_pred, pos_label=1)
```

In [ ]:
print(f'The recall of the class of not-canceled is {metrics.recall_score(titTrainY,titTrainYPre,pos_label=0)}')

In [ ]:
print(f'The recall of the class of canceled is {metrics.recall_score(titTrainY,titDt.predict(titTrainX),pos_label=1)}')

### Precision Score

```python
from sklearn import metrics
metrics.precision_score(y_true, y_pred, pos_label=1)
```

In [ ]:
print(f'The precision of the class of not-canceled is {metrics.precision_score(titTrainY,titTrainYPre,pos_label=0)}')

In [ ]:
print(f'The precision of the class of canceled is {metrics.precision_score(titTrainY,titTrainYPre,pos_label=1)}')

### $F_1$ Score

- Considers both recall and precision; it is the harmonic mean of recall and precision

```python
from sklearn import metrics
metrics.f1_score(y_true, y_pred, pos_label=1)
```

In [ ]:
print(f'The F1_score of the class of not-canceled is {metrics.f1_score(titTrainY,titTrainYPre,pos_label=0)}')

In [ ]:
print(f'The F1_score of the class of canceled is {metrics.f1_score(titTrainY,titTrainYPre,pos_label=1)}')

### P-R Curve

- A curve formed by precision and recall
- x-axis is recall, y-axis is precision

In [ ]:
from sklearn.metrics import precision_recall_curve
y_true = np.array([0, 0, 1, 1])
y_scores = np.array([0.1, 0.4, 0.35, 0.8])
precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
f'Precision {precision}'
f'Recall {recall}'
f'Thresholds {thresholds}'

```python
sklearn.metrics.plot_precision_recall_curve(estimator, X, y, pos_label)
```
- `estimator`: the trained classification model
- `X`: the input attributes
- `y`: the label values (binary labels)
- `pos_label`: `int` or `str`, the specified class, defaults to 1

In [ ]:
metrics.plot_precision_recall_curve(titDt,titTrainX,titTrainY,pos_label=0)

- `AP`: average precision score, the area under the P-R curve; the larger the better

### ROC Curve

- Receiver Operating Characteristic (ROC) Curve, a curve formed by the "True Positive Rate" and the "False Positive Rate"
$$
TPR = \frac{TP}{TP+FN}\\
FPR = \frac{FP}{FP+TN}
$$
- the x-axis is FPR, and the y-axis is TPR

```python
sklearn.metrics.plot_roc_curve(estimator, X, y, pos_label)
```
- `estimator`: the trained classification model
- `X`: the input attributes
- `y`: the label values (binary labels)
- `pos_label`: `int` or `str`, the specified class, defaults to 1

In [ ]:
metrics.plot_roc_curve(titDt,titTrainX,titTrainY,pos_label=0)

- `AUC`: the area under the ROC curve; the larger the better

- How to choose between the two curves
    - ROC: suitable when the classes are balanced
    - P-R: suitable when the classes are imbalanced

### Classification Performance on the Test Set

###### Generating class predictions for the test set

In [ ]:
titTestYPre = titDt.predict(titTestX)

###### Classification Performance Metrics

In [ ]:
print(f'The classification accuracy on the test set is {metrics.accuracy_score(titTestY,titTestYPre)}')

In [ ]:
print(f'The recall of the class of not-canceled on the test set is {metrics.recall_score(titTestY,titTestYPre,pos_label=0)}')

In [ ]:
print(f'The recall of the class of canceled on the test set is {metrics.recall_score(titTestY,titTestYPre,pos_label=1)}')

###### Classification Performance Metrics - Continued

In [ ]:
print(f'The precision of the class of not-canceled on the test set is {metrics.precision_score(titTestY,titTestYPre,pos_label=0)}')

In [ ]:
print(f'The precision of the class of canceled on the test set is {metrics.precision_score(titTestY,titTestYPre,pos_label=1)}')

In [ ]:
print(f'The F1_score of the class of not-canceled on the test set is {metrics.f1_score(titTestY,titTestYPre,pos_label=0)}')

In [ ]:
print(f'The F1_score of the class of canceled on the test set is {metrics.f1_score(titTestY,titTestYPre,pos_label=1)}')

In [ ]:
metrics.plot_precision_recall_curve(titDt,titTestX,titTestY)

In [ ]:
metrics.plot_roc_curve(titDt,titTestX,titTestY)

<p class="alert alert-danger">The decision tree model performs well on the <strong>training</strong> set, but its classification performance on the <strong>test</strong> set is only average.</p>

## Model Overfitting

- **Training error**: the fraction of the samples misclassified on the training set

- **Generalization error**: the expected error of the classification model on unseen records

> In loss terms: **training error = 0-1 loss on the training set**, **generalization error = 0-1 loss on unseen data**. Every algorithm minimizes a loss on the training set; **overfitting** is when training loss keeps dropping while generalization loss rises.

#### Model Underfitting

#### Overfitting

- A classification model that matches the training set too closely can achieve a very low training error, but its generalization error will be higher than its training error; this phenomenon is called model overfitting

- As the **complexity** of the classification model increases, the training error keeps decreasing, but such a model will match the noise present in the training set, leading to a higher generalization error

### Pruning a Decision Tree

- Why pruning is needed?
    + the generated decision tree is too complex and overfits the training set
    + the decision tree algorithm does not take into account the noise present in the data

- Pruning methods
    + forward pruning (pre-pruning)
    + post-pruning

#### Forward (Pre-)Pruning

<p class="alert alert-list">Prune the tree by stopping its construction early.</p>

- Stop the growth of the tree once it reaches a certain height
- Stop the growth of the tree when the number of samples reaching a node is smaller than some threshold

```python
tree.DecisionTreeClassifier(max_depth=None, min_samples_split=2, min_samples_leaf=1)
```

- `max_depth`: `int` or `None`, the maximum depth of the tree. If `None`, then all leaves contain only pure classes, or all leaves contain fewer than `min_samples_leaf` samples
    + if the value is too large, the algorithm **overfits** the training set; if too small, it hinders the algorithm from learning the data
    + the recommended initial value is 3; first inspect how well the generated tree fits the data, and then decide whether to increase the depth

- `min_samples_split`: `int` or `float`, the minimum number of samples required to split an internal node
    + as an `int`, `min_samples_split` is the minimum, defaulting to 2 samples
    + as `float`, the fraction of the whole set of samples, and `ceil(min_samples_split * n_samples)` is the minimum
    + the larger the value, the fewer the branches of the tree, achieving a degree of forward-pruning

- `min_samples_leaf`: `int` or `float`, the minimum number of samples required in each leaf node
    + as `int`, `min_samples_leaf` is the minimum, defaulting to 1 sample
    + as `float`, the fraction of the whole set of samples; the `ceil(min_samples_leaf * n_samples)` is the minimum
    + the larger the value, the fewer the branches of the tree, achieving a degree of forward-pruning

#### Applying Forward Pruning to the Hotel Booking Decision Tree

##### Decision Tree Model

In [ ]:
titDtForp = tree.DecisionTreeClassifier(max_depth=5,min_samples_split=10,min_samples_leaf=10)

##### Training

In [ ]:
titDtForp.fit(titTrainX,titTrainY)

In [ ]:
titDotForp = tree.export_graphviz(titDtForp,feature_names=list(titTrainX.columns),class_names=['Not canceled','Canceled'],filled=True)
titDtForpGraph = graphviz.Source(titDotForp)
titDtForpGraph

##### Performance of the Decision Tree on the Training Set

In [ ]:
titTrainYPreForp = titDtForp.predict(titTrainX)

In [ ]:
print(f'The F1_score of the forward-pruned decision tree on the training set is {metrics.f1_score(titTrainY,titTrainYPreForp)}')

In [ ]:
metrics.plot_roc_curve(titDtForp,titTrainX,titTrainY)

##### Comparison with the Unpruned Decision Tree on the Training Set

In [ ]:
titTrainDisp = metrics.plot_roc_curve(titDtForp,titTrainX,titTrainY,label='forward-pruning')
metrics.plot_roc_curve(titDt,titTrainX,titTrainY,ax=titTrainDisp.ax_,label='original')
# titTrainDisp.ax_: obtain the plotting axis

##### Performance of the Decision Tree on the Test Set

In [ ]:
titTestYPreForp = titDtForp.predict(titTestX)

In [ ]:
print(f'The F1_score of the forward-pruned decision tree on the training set is {metrics.f1_score(titTestY,titTestYPreForp)}')

In [ ]:
metrics.plot_roc_curve(titDtForp,titTestX,titTestY)

##### Comparison with the Unpruned Decision Tree on the Test Set

In [ ]:
titTestDisp = metrics.plot_roc_curve(titDtForp,titTestX,titTestY,label='forward-pruning')
metrics.plot_roc_curve(titDt,titTestX,titTestY,ax=titTestDisp.ax_,label='original')
# titTestDisp.ax_: obtain the plotting axis

#### Post-Pruning

<p class="alert alert-danger">Construct the complete decision tree, and then replace the subtrees of the nodes that are not confident enough with leaf nodes; the class assigned to that leaf node is the class to which the majority of the samples in the replaced subtree belong.</p>

```python
tree.DecisionTreeClassifier(ccp_alpha=None)
```
- `ccp_alpha`: a non-negative float; use cost-complexity pruning and keep those subtrees whose cost-complexity value is lower than this parameter

#### Applying Post-Pruning to the Hotel Booking Decision Tree

##### Building the Post-Pruned Model

In [ ]:
titDtPostp = tree.DecisionTreeClassifier(ccp_alpha=0.035,random_state=10)

##### Training the Decision Tree

In [ ]:
titDtPostp.fit(titTrainX,titTrainY)

##### Visualizing the Decision Tree

In [ ]:
titDotPostp = tree.export_graphviz(titDtPostp,feature_names=list(titTrainX.columns),class_names=['Not canceled','Canceled'],filled=True)
titDotPostpGraph = graphviz.Source(titDotPostp)
titDotPostpGraph

##### Classification Performance on the Training Set

In [ ]:
titTrainYPrePostp = titDtPostp.predict(titTrainX)

In [ ]:
print(f'The F1_score of the post-pruned decision tree on the training set is {metrics.f1_score(titTrainY,titTrainYPrePostp)}')

In [ ]:
titTrainDisp = metrics.plot_roc_curve(titDt,titTrainX,titTrainY,label='original')
metrics.plot_roc_curve(titDtForp,titTrainX,titTrainY,label='forward-pruning',ax=titTrainDisp.ax_)
metrics.plot_roc_curve(titDtPostp,titTrainX,titTrainY,label='post-pruning',ax=titTrainDisp.ax_)

##### Classification Performance on the Test Set

In [ ]:
titTestYPrePostp = titDtPostp.predict(titTestX)

In [ ]:
print(f'The F1_score of the post-pruned decision tree on the test set is {metrics.f1_score(titTestY,titTestYPrePostp)}')

In [ ]:
titTestDisp = metrics.plot_roc_curve(titDt,titTestX,titTestY,label='original')
metrics.plot_roc_curve(titDtForp,titTestX,titTestY,label='forward-pruning',ax=titTestDisp.ax_)
metrics.plot_roc_curve(titDtPostp,titTestX,titTestY,label='post-pruning',ax=titTestDisp.ax_)

##### How to Choose $ccp\_alpha$?

```python
dt.cost_complexity_pruning_path(self, X, y)
```
- returns the computation process of cost-complexity pruning, a dictionary including the `ccp_alpha` array and the `impurities` array
- `X`: the predictive attributes of the training set
- `y`: the list of class labels

###### Obtaining the Pruning `ccp_alpha`

In [ ]:
ccp_path = titDt.cost_complexity_pruning_path(titTrainX, titTrainY)
alphas = ccp_path['ccp_alphas']
ccp_path

###### Generating a list of decision trees with different `ccp_alpha`

In [ ]:
dts = []
for ccp_alpha in alphas[:-1]:
    # alphas[:-1] removes the maximum, since it contains only one node
    dt = tree.DecisionTreeClassifier(random_state=10, ccp_alpha=ccp_alpha)
    dt.fit(titTrainX, titTrainY)
    dts.append(dt)

###### Computing the f1-score of each tree on the training and test sets

In [ ]:
trainScoreLst = [metrics.f1_score(titTrainY,dt.predict(titTrainX)) for dt in dts]
testScoreLst = [metrics.f1_score(titTestY,dt.predict(titTestX)) for dt in dts]

In [ ]:
alphaTestDf = pd.DataFrame({'a':alphas[:-1],'train':trainScoreLst,'test':testScoreLst})
ax = alphaTestDf.plot(x='a',y='train',kind='line',figsize=(12,6),marker='o')
alphaTestDf.plot(x='a',y='test',kind='line',marker='d',ax=ax)
ax.set(title='ccp_alpha v.s. accuracy',xlabel='ccp_alpha',ylabel='accuracy')

[Possible plot data-point markers https://matplotlib.org/3.2.1/api/markers_api.html](https://matplotlib.org/3.2.1/api/markers_api.html)

##### Selecting the Best `ccp_alpha`

In [ ]:
alphaTestDf.loc[(alphaTestDf['train']>=0.8)& (alphaTestDf['test']==alphaTestDf['test'].max()),:]

##### Visualizing the Decision Tree

In [ ]:
print(tree.export_text(dts[40],feature_names=list(titTrainX.columns)))

## Random Forest

### Ensemble Learning

<dl class="row alert-danger">
    <dt class="col-md-3">Ensemble learning</dt>
    <dd class="col-md-9">Complete the learning task by combining and combining multiple classifiers.</dd>
</dl>

- Typical methods
    - bagging and random forest
    - boosting: an algorithm that boosts weak classifiers into strong classifiers

### Bagging

<dl class="row alert-info">
    <dt class="col-md-3">Bootstrap sampling</dt>
    <dd class="col-md-9">The random sampling method with replacement.</dd>
</dl>

- Principle: if we have $N$ independent and identically distributed ($iid$) samples, each with variance $\sigma^2$, then the variance of the sample mean is $\frac{\sigma^2}{N}$

- The bagging process
    1. generate $B$ bootstrap samples;
    2. train a classifier $\hat{f}_b(\boldsymbol{\rm x})$ on each sample;
    3. combine the classifiers to obtain the final classifier
    $$
        \hat{f}_{\text{avg}}(\boldsymbol{\rm x})=\frac{1}{B}\sum_{b=1}^B\hat{f}_b(\boldsymbol{\rm x})
    $$

### Random Forest

- Feature bagging, to reduce the correlation between different decision trees

- Combine multiple decision trees, determine the class of a sample by voting, so that the overall model obtains good accuracy while suppressing overfitting

<center><img src="./img/classification/randomForest.jpg" width=100%></center>

### Building a Random Forest Model

```python
from sklearn import ensemble
ensemble.RandomForestClassifier()
```

In [ ]:
rbRandTree = ensemble.RandomForestClassifier(random_state=10)

In [ ]:
rbRandTree.fit(titTrainX,titTrainY)

### Testing the Classification Performance on the Training Set

In [ ]:
rbRandTrainYPre = rbRandTree.predict(titTrainX)

In [ ]:
metrics.accuracy_score(rbRandTrainYPre,titTrainY)

In [ ]:
metrics.f1_score(rbRandTrainYPre,titTrainY,pos_label=1)

In [ ]:
metrics.f1_score(rbRandTrainYPre,titTrainY,pos_label=0)

### Testing the Classification Performance on the Test Set

In [ ]:
rbRandTestYPre = rbRandTree.predict(titTestX)

In [ ]:
metrics.accuracy_score(rbRandTestYPre,titTestY)

In [ ]:
metrics.f1_score(rbRandTestYPre,titTestY,pos_label=1)

In [ ]:
metrics.f1_score(rbRandTestYPre,titTestY,pos_label=0)

- Comparison of the ROC curves of the random forest with the original decision tree, the forward-pruned decision tree, and the post-pruned decision tree

In [ ]:
randTreeDisp = metrics.plot_roc_curve(rbRandTree,titTestX,titTestY,label='random forest')
metrics.plot_roc_curve(titDt,titTestX,titTestY,label='original',ax=randTreeDisp.ax_)
metrics.plot_roc_curve(titDtForp,titTestX,titTestY,label='forward pruning',ax=randTreeDisp.ax_)
metrics.plot_roc_curve(dts[40],titTestX,titTestY,label='post pruning',ax=randTreeDisp.ax_)